In [ ]:
# Install packages

!pip install -q datasets transformers torch scikit-learn pandas tqdm

In [ ]:

# 2. Import libraries

import os
import re
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [ ]:
# 3. Config


DATASET_NAME = "tattabio/ec_classification"

ID_COL = "Entry"
LABEL_COL = "Label"
SEQ_COL = "Sequence"

MODEL_NAME = "facebook/esm2_t6_8M_UR50D"

BATCH_SIZE = 8
MAX_LENGTH = 1024
RANDOM_STATE = 42

OUTPUT_DIR = "dgeb_embedding_baseline_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
#  4. Load DGEB EC dataset


dataset = load_dataset(DATASET_NAME)

train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())
display(test_df.head())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/485 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/285k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/87.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/512 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/128 [00:00<?, ? examples/s]

Train shape: (512, 3)
Test shape: (128, 3)


,Entry,Label,Sequence
0,Q9LQC0,1.14.14.18,MATSRLNASCRFPASRRLDCESYVSLRAKTVTIRYVRTIAAPRRHL...
1,A0AFT8,1.14.14.18,MIIVTNTIKVEKGAAEHVIRQFTGANGDGHPTKDIAEVEGFLGFEL...
2,P74133,1.14.14.18,MTNLAQKLRYGTQQSHTLAENTAYMKCFLKGIVEREPFRQLLANLY...
3,O31534,1.14.14.18,MFVQLRKMTVKEGFADKVIERFSAEGIIEKQEGLIDVTVLEKNVRR...
4,P34697,1.15.1.1,MFMNLLTQVSNAIFPQVEAAQKMSNRAVAVLRGETVTGTIWITQKS...


,Entry,Label,Sequence
0,A0A0H2XEA6,1.14.14.18,MMQALGQGHIDADTYAQVLRRHHRLLAGFEEQLSDWLVTLVGSGWQ...
1,Q5AD07,1.15.1.1,MKYLSIFLLATFALAGDAPISTDSKGSPSLIAKFEKTSKSNIEGTI...
2,O74831,1.16.3.1,MQSLRAAFRRRTPIFLKPYEFSTNVFGLRCRYYSQVRHNGALTDLE...
3,O46310,1.17.4.1,MKPVAAGAEVLPADKVAQGTNAEEEPLQQENPFRYVLFPIQYHDIW...
4,Q8PU58,1.5.98.3,MIGDTMSGIIDSYIPVAIFLAVGLIMPPMTMFMVKQLSPRSKAASK...


In [ ]:
# 5. Clean protein sequences

def clean_protein_sequence(seq):
    if pd.isna(seq):
        return ""

    seq = str(seq).upper()
    seq = seq.replace(" ", "").replace("\n", "")

    # Replace rare amino acids with X
    seq = re.sub(r"[UZOB]", "X", seq)

    # Keep only letters
    seq = re.sub(r"[^A-Z]", "", seq)

    return seq


train_df[SEQ_COL] = train_df[SEQ_COL].apply(clean_protein_sequence)
test_df[SEQ_COL] = test_df[SEQ_COL].apply(clean_protein_sequence)

train_df = train_df[train_df[SEQ_COL].str.len() > 0].reset_index(drop=True)
test_df = test_df[test_df[SEQ_COL].str.len() > 0].reset_index(drop=True)

print("After cleaning:")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

After cleaning:
Train shape: (512, 3)
Test shape: (128, 3)


In [ ]:
# 6. Encode labels


label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(train_df[LABEL_COL])
y_test = label_encoder.transform(test_df[LABEL_COL])

print("Number of classes:", len(label_encoder.classes_))
print("Example labels:", label_encoder.classes_[:10])

Number of classes: 128
Example labels: ['1.14.14.18' '1.15.1.1' '1.16.3.1' '1.17.4.1' '1.5.98.3' '1.8.3.2'
 '2.1.1.354' '2.1.1.360' '2.1.1.37' '2.1.1.72']


In [ ]:
# 7. Load pretrained ESM-2 model


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

model = model.to(device)
model.eval()

print("Loaded model:", MODEL_NAME)

Using device: cuda


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded model: facebook/esm2_t6_8M_UR50D


In [ ]:
# 8. Embedding function


@torch.no_grad()
def embed_sequences(sequences, batch_size=8, max_length=1024):
    all_embeddings = []

    for start in tqdm(range(0, len(sequences), batch_size), desc="Embedding sequences"):
        batch_sequences = sequences[start:start + batch_size]

        encoded = tokenizer(
            batch_sequences,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state

        # mean pooling, ignoring padding
        mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        summed = torch.sum(last_hidden * mask, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)
        mean_pooled = summed / counts

        all_embeddings.append(mean_pooled.cpu().numpy())

    return np.vstack(all_embeddings)

In [ ]:
# 9. Generate or load embeddings

train_emb_path = os.path.join(OUTPUT_DIR, "train_esm2_embeddings.npy")
test_emb_path = os.path.join(OUTPUT_DIR, "test_esm2_embeddings.npy")

if os.path.exists(train_emb_path) and os.path.exists(test_emb_path):
    print("Loading cached embeddings...")
    X_train = np.load(train_emb_path)
    X_test = np.load(test_emb_path)
else:
    print("Generating train embeddings...")
    X_train = embed_sequences(
        train_df[SEQ_COL].tolist(),
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH
    )

    print("Generating test embeddings...")
    X_test = embed_sequences(
        test_df[SEQ_COL].tolist(),
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH
    )

    np.save(train_emb_path, X_train)
    np.save(test_emb_path, X_test)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Generating train embeddings...


Embedding sequences:   0%|          | 0/64 [00:00<?, ?it/s]

Generating test embeddings...


Embedding sequences:   0%|          | 0/16 [00:00<?, ?it/s]

X_train shape: (512, 320)
X_test shape: (128, 320)


In [ ]:
# 10. Train Logistic Regression


clf = LogisticRegression(
    max_iter=5000,
    C=1.0,
    solver="lbfgs",
    multi_class="auto",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

clf.fit(X_train, y_train)

print("Training finished.")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Training finished.


In [ ]:
# 11. Evaluate


y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
weighted_f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print("ESM-2 Embedding + Logistic Regression Results")
print("---------------------------------------------")
print(f"Accuracy:    {accuracy:.4f}")
print(f"Macro F1:    {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

ESM-2 Embedding + Logistic Regression Results
---------------------------------------------
Accuracy:    0.4453
Macro F1:    0.3937
Weighted F1: 0.3937


In [ ]:
# 12. Classification report

print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_,
    zero_division=0
))

              precision    recall  f1-score   support

  1.14.14.18       0.00      0.00      0.00         1
    1.15.1.1       1.00      1.00      1.00         1
    1.16.3.1       0.00      0.00      0.00         1
    1.17.4.1       1.00      1.00      1.00         1
    1.5.98.3       0.00      0.00      0.00         1
     1.8.3.2       0.00      0.00      0.00         1
   2.1.1.354       0.00      0.00      0.00         1
   2.1.1.360       0.00      0.00      0.00         1
    2.1.1.37       0.50      1.00      0.67         1
    2.1.1.72       0.00      0.00      0.00         1
    2.1.1.77       1.00      1.00      1.00         1
    2.1.1.86       0.50      1.00      0.67         1
    2.1.3.15       0.00      0.00      0.00         1
   2.3.1.225       0.00      0.00      0.00         1
   2.3.1.269       1.00      1.00      1.00         1
    2.3.1.48       0.00      0.00      0.00         1
    2.3.1.51       1.00      1.00      1.00         1
    2.3.2.23       0.00    

In [ ]:
# 13. Save predictions


pred_labels = label_encoder.inverse_transform(y_pred)
true_labels = label_encoder.inverse_transform(y_test)

results_df = pd.DataFrame({
    "Entry": test_df[ID_COL].values,
    "True_Label": true_labels,
    "Predicted_Label": pred_labels,
    "Correct": true_labels == pred_labels
})

results_path = os.path.join(OUTPUT_DIR, "esm2_embedding_predictions.csv")
results_df.to_csv(results_path, index=False)

print("Saved predictions to:", results_path)
display(results_df.head(20))

Saved predictions to: dgeb_embedding_baseline_outputs/esm2_embedding_predictions.csv


,Entry,True_Label,Predicted_Label,Correct
0,A0A0H2XEA6,1.14.14.18,5.4.99.5,False
1,Q5AD07,1.15.1.1,1.15.1.1,True
2,O74831,1.16.3.1,6.5.1.3,False
3,O46310,1.17.4.1,1.17.4.1,True
4,Q8PU58,1.5.98.3,7.1.1.9,False
5,Q5UQV6,1.8.3.2,2.7.7.6,False
6,Q8IRW8,2.1.1.354,2.1.1.360,False
7,Q6BTC8,2.1.1.360,2.7.1.67,False
8,P0DOY1,2.1.1.37,2.1.1.37,True
9,Q09956,2.1.1.72,2.1.1.37,False


In [ ]:
# 14. Save summary result


summary_df = pd.DataFrame([{
    "Method": "ESM-2 Embedding + Logistic Regression",
    "Accuracy": accuracy,
    "Macro_F1": macro_f1,
    "Weighted_F1": weighted_f1,
    "Model": MODEL_NAME,
    "Train_Samples": len(train_df),
    "Test_Samples": len(test_df),
    "Num_Classes": len(label_encoder.classes_)
}])

summary_path = os.path.join(OUTPUT_DIR, "esm2_embedding_summary.csv")
summary_df.to_csv(summary_path, index=False)

print("Saved summary to:", summary_path)
display(summary_df)

Saved summary to: dgeb_embedding_baseline_outputs/esm2_embedding_summary.csv


,Method,Accuracy,Macro_F1,Weighted_F1,Model,Train_Samples,Test_Samples,Num_Classes
0,ESM-2 Embedding + Logistic Regression,0.445312,0.39375,0.39375,facebook/esm2_t6_8M_UR50D,512,128,128
